<div style='text-align:center;padding:26px;background:linear-gradient(135deg,#0f2027 0%,#2c5364 100%);border-radius:14px'>
<h1 style='color:#fff;margin:0 0 6px 0'>Qwen-Image-2.1</h1>
<p style='color:#cfe3ef;margin:0'>7B INT8 &middot; Wan2GP engine &middot; Kaggle GPU T4 x2</p>
</div>

**Licence modele :** `qwen-research` (Qwen). Poids quantifies par DeepBeepMeep.

**IMPORTANT - poids persistes :** les 18,7 Go sont stockes dans `/kaggle/working/weights/`,
donc ils sont **conserves dans l'output du kernel**. On pourra en faire un dataset Kaggle
une fois, puis attacher ce dataset aux versions suivantes pour supprimer le telechchargement.

**Settings :** Accelerator = **GPU T4 x2**, Internet = **On**.

`--smoke-test` : generation de controle puis arret. `--no-test` : ouvre l'UI directement.

## Step 1 - Environment

In [ ]:
import os, gc, subprocess, psutil

print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} Go")
os.system("echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1")
gc.collect()
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True, timeout=30).stdout)
print("environment ok")

## Step 2 - Installer Wan2GP

Le moteur (`models/qwen21`) est une implementation de Qwen-Image-2.1 dans Wan2GP.

In [ ]:
!git clone --depth 1 https://github.com/DeepBeepMeep/Wan2GP.git
!pip install --timeout 180 --retries 5 -q -r Wan2GP/requirements.txt
!pip install --timeout 180 --retries 5 -q mmgp gradio huggingface_hub
print("install ok")

## Step 3 - Poids (~18,7 Go, conserves dans l'output)

Verifie l'espace disque avant de telecharger, puis lie les fichiers dans `Wan2GP/models` par symlink.

In [ ]:
import os, shutil, json, urllib.request
from huggingface_hub import hf_hub_download

REPO = "DeepBeepMeep/Qwen_image_2"
ENC_REPO = "DeepBeepMeep/Ideogram4"

# Les poids vont dans /kaggle/working (PERMANENT, sauvegarde comme output)
# et pas /kaggle/tmp (efface a la fin du run). Ca permet de creer un dataset
# Kaggle une seule fois, et d'attacher ce dataset aux versions suivantes :
# plus aucun telechchargement, on passe de 20 min a ~2 min par iteration.
WEIGHTS = "/kaggle/working/weights"
MODEL_DIR = "Wan2GP/models"
os.makedirs(WEIGHTS, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# --- bilan disque avant de telecharger ---
SPEC = [
    (REPO, "qwen_image_21_7B_int8_convrot.safetensors"),
    (ENC_REPO, "Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_int8_convrot.safetensors"),
    (ENC_REPO, "Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_vision_bf16.safetensors"),
    (REPO, "qwen_image_21/qwen_image_21_vae.safetensors"),
] + [(ENC_REPO, f"Qwen3-VL-8B-Instruct/{f}") for f in [
    "config_legacy.json", "tokenizer_legacy.json", "tokenizer_config_legacy.json",
    "preprocessor_config.json", "chat_template.jinja", "merges.txt",
    "vocab.json", "added_tokens.json", "special_tokens_map.json",
    "video_preprocessor_config.json"]] + [(REPO, f"qwen_image_21/{f}") for f in [
    "vae_config.json", "scheduler_config.json", "transformer_config.json"]]

def size_of(repo, fn):
    try:
        u = f"https://huggingface.co/api/models/{repo}/tree/main/{fn}"
        for e in json.load(urllib.request.urlopen(u, timeout=30)):
            return (e.get("lfs") or {}).get("size") or e.get("size", 0)
    except Exception:
        return 0
    return 0

need = sum(size_of(r, f) for r, f in SPEC)
st = os.statvfs("/kaggle/working")
free = st.f_bavail * st.f_frsize / 1e9
print(f"poids requis : {need/1e9:.1f} Go")
print(f"espace libre /kaggle/working : {free:.1f} Go")
if free < need / 1e9 + 1.0:
    raise SystemExit(f"ESPACE INSUFFISANT : {need/1e9 + 1.0:.1f} Go necessaires, {free:.1f} Go libres")


def fetch(repo, filename, dest_dir):
    dest = os.path.join(dest_dir, os.path.basename(filename))
    if os.path.exists(dest):
        print("  deja la  ", os.path.basename(filename))
        return dest
    local = hf_hub_download(repo_id=repo, filename=filename, local_dir=WEIGHTS,
                            local_dir_use_symlinks=False)
    os.makedirs(dest_dir, exist_ok=True)
    if os.path.lexists(dest):
        os.remove(dest)
    os.symlink(local, dest)
    assert os.path.exists(dest), f"symlink casse: {dest}"
    return dest


ENC_DIR = os.path.join(MODEL_DIR, "Qwen3-VL-8B-Instruct")
VAE_DIR = os.path.join(MODEL_DIR, "qwen_image_21")

print("\ntelechargement des poids (~18.7 Go)...")
for r, f in SPEC:
    dest_dir = MODEL_DIR
    if "Qwen3-VL-8B-Instruct" in f:
        dest_dir = ENC_DIR
    elif "qwen_image_21/" in f:
        dest_dir = VAE_DIR
    fetch(r, f, dest_dir)
    print("  ok", os.path.basename(f))

for d in [os.path.join(WEIGHTS, ".cache"), "/kaggle/tmp/hf_home"]:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)

missing = []
for r, f in SPEC:
    dest_dir = MODEL_DIR
    if "Qwen3-VL-8B-Instruct" in f:
        dest_dir = ENC_DIR
    elif "qwen_image_21/" in f:
        dest_dir = VAE_DIR
    p = os.path.join(dest_dir, os.path.basename(f))
    if not os.path.exists(p):
        missing.append(p)
print("\n--- verification ---")
print("fichiers manquants:", missing if missing else "aucun")
if missing:
    raise FileNotFoundError(", ".join(missing))
os.system("du -sh /kaggle/working/weights")
os.system("df -h /kaggle/working")
print("poids prets ET conserves dans l'output du kernel")


## Step 4 - Script de generation

Trois correctifs par rapport au portage direct : gel des gradients avant chargement INT8, enregistrement du qtype `int8_convrot`, et dtype fp16 pour le T4.

In [ ]:
%%writefile run_qwen_image.py
# Qwen-Image-2.1 (7B) - standalone text-to-image for Kaggle T4 x2
# Engine: Wan2GP (models/qwen21). Run with --smoke-test for a headless single-image check.
import gc
import os
import sys
import time
import traceback
import zipfile
import tempfile

import numpy as np
import torch
from PIL import Image

SMOKE = "--smoke-test" in sys.argv
SKIP_TEST = "--no-test" in sys.argv
OUT_DIR = "/kaggle/working/outputs" if os.path.isdir("/kaggle/working") else "outputs"
os.makedirs(OUT_DIR, exist_ok=True)


def log(*a):
    print(*a, flush=True)


log("=" * 60)
log("Qwen-Image-2.1 | smoke-test =", SMOKE)
log("torch", torch.__version__, "| cuda", torch.version.cuda)
if torch.cuda.is_available():
    log("GPUs:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        log(f"  [{i}] {p.name} {p.total_memory / 1024**3:.1f} GB")
else:
    log("!! no CUDA visible")

# ---- bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from shared.utils import files_locator as fl
from models.qwen.qwen_handler import family_handler

# --- correctif : geler les poids AVANT le chargement quantifie ---
# qwen21_handler cree l'encodeur sous init_empty_weights() -> requires_grad=True,
# puis appelle load_model_data(..., assign=True) avec des tenseurs INT8.
# PyTorch refuse d'assigner un tenseur entier a un Parameter qui exige le grad :
#   RuntimeError: Only Tensors of floating point and complex dtype can require gradients
# Le handler fait .requires_grad_(False) APRES le chargement, donc trop tard.
# On enveloppe load_model_data pour geler systematiquement avant.
def _patch_freeze_before_load():
    from mmgp import offload as _off
    if getattr(_off, "_frozen_patched", False):
        return
    _orig = _off.load_model_data

    def _frozen(model, *a, **kw):
        try:
            model.eval()
            model.requires_grad_(False)
        except Exception:
            pass
        return _orig(model, *a, **kw)

    _off.load_model_data = _frozen
    _off._frozen_patched = True
    log("patch load_model_data : gel avant chargement applique")


try:
    _patch_freeze_before_load()
except Exception:
    log("patch load_model_data echoue (on continue):")
    traceback.print_exc()

# --- correctif 2 : enregistrer le qtype "int8_convrot" ---
# wgp.py fait cet enregistrement au demarrage, mais on importe family_handler
# directement (comme le notebook Krea-2) -> on saute wgp.py entierement.
# Sans ca, les poids *_int8_convrot.safetensors restent des tenseurs int8 bruts
# et F.linear(entree_bf16, poids_int8) leve :
#   RuntimeError: expected mat1 and mat2 to have the same dtype
# Krea-2 echappait a ca car il utilise du quanto int8, gere nativement par
# optimum.quanto. Le format convrot est propre a Wan2GP et exige l'enregistrement.
try:
    from mmgp import quant_router

    _QTYPE_MODULES = [
        "shared.qtypes.scaled_fp8",
        "shared.qtypes.nvfp4",
        "shared.qtypes.bnb_nf4",
        "shared.qtypes.nunchaku_int4",
        "shared.qtypes.nunchaku_fp4",
        "shared.qtypes.asym_w4a8_int8",
        "shared.qtypes.int8_convrot",
        "shared.qtypes.gguf",
    ]
    for _m in _QTYPE_MODULES:
        quant_router.register_handler(_m)
    log("qtypes enregistres :", quant_router._HANDLER_MODULES)
except Exception:
    log("enregistrement qtype echoue:")
    traceback.print_exc()

fl.set_checkpoints_paths(["models", "ckpts", "."])

BASE_MODEL_TYPE = "qwen_image_21_7B"
model_def = family_handler.query_model_def(BASE_MODEL_TYPE, {})

TRANSFORMER = "models/qwen_image_21_7B_int8_convrot.safetensors"
ENCODER = "models/Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_int8_convrot.safetensors"

for p in (TRANSFORMER, ENCODER,
          "models/qwen_image_21/qwen_image_21_vae.safetensors",
          "models/qwen_image_21/vae_config.json",
          "models/qwen_image_21/scheduler_config.json",
          "models/Qwen3-VL-8B-Instruct/config_legacy.json",
          "models/Qwen3-VL-8B-Instruct/tokenizer_legacy.json",
          "models/Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_vision_bf16.safetensors"):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing: {p}")
log("all model files present")

log("loading model (INT8 transformer + INT8 Qwen3-VL-8B encoder)...")
t0 = time.time()
pipeline, components = family_handler.load_model(
    model_filename=[TRANSFORMER],          # qwen21 expects a LIST
    model_type=BASE_MODEL_TYPE,
    base_model_type=BASE_MODEL_TYPE,
    model_def=model_def,
    quantizeTransformer=False,
    text_encoder_filename=ENCODER,
    VAE_dtype=torch.float32,               # fp16 overflows on real latents (handler warning)
)
log(f"model loaded in {time.time() - t0:.1f}s")

# ---- correction dtype : T4 (Turing) n'a pas de Tensor Cores bf16 ----
# qwen21_handler pose, sur TOUS les modules :
#     module._lock_dtype = None ; component._convertWeightsFloatTo = None
# Or mmgp teste hasattr(module, "_lock_dtype") -> True meme pour None :
#   - le scan "Scanning Sizes" saute toute normalisation de dtype
#   - hook_change_module prend dtype=None, donc n'aligne pas les entrees
# Resultat : entrees en bfloat16, poids en int8 ->
#   RuntimeError: expected mat1 and mat2 to have the same dtype
#                  (c10::BFloat16 != signed char)
# krea2_handler ne fait PAS ces assignations, c'est donc la que le chemin
#.dtype de mmgp fonctionne. On les annule et on force le fp16 (rapide sur T4).
def _fix_dtypes_for_t4(components, target):
    for name, comp in components.items():
        if name == "vae":
            continue  # le VAE doit rester en fp32 (fp16 overflow sur les latents)
        n_lock = sum(1 for m in comp.modules() if hasattr(m, "_lock_dtype"))
        for attr in ("_convertWeightsFloatTo", "_model_dtype"):
            if hasattr(comp, attr):
                delattr(comp, attr)
        for m in comp.modules():
            if hasattr(m, "_lock_dtype"):
                del m._lock_dtype
        n_cast = 0
        for p in comp.parameters():
            if p.is_floating_point() and p.data.dtype != target:
                p.data = p.data.to(target)
                n_cast += 1
        log(f"  dtype {name}: {n_lock} _lock_dtype supprimes, {n_cast} params -> {target}")


try:
    _fix_dtypes_for_t4(components, torch.float16)
except Exception:
    log("correction dtype echouee:")
    traceback.print_exc()

# ---- mmgp memory profile (T4 x2 = 32 GB VRAM) ----
try:
    from mmgp import offload

    offload.profile(
        components,
        profile_no=2,
        quantizeTransformer=False,
        convertWeightsFloatTo=torch.float16,
        pinnedMemory=True,
        asyncTransfers=True,
        budgets={"transformer": 12000, "text_encoder": 9500, "vae": 1500, "*": 800},
    )
    offload.shared_state["_attention"] = "sdpa"
    log("mmgp profile applied")
except Exception:
    log("mmgp profile skipped:")
    traceback.print_exc()


def to_pil(result):
    # generate() returns uint8 [channels, batch, H, W]
    arr = result[:, 0]
    if arr.shape[0] == 4:
        arr = arr[:3]
    return Image.fromarray(arr.permute(1, 2, 0).cpu().numpy())


def run_one(prompt, steps, width, height, seed, guide, negative=None, progress=None):
    def cb(step_idx, latent, is_start, **kw):
        if progress:
            progress((step_idx + 1) / max(steps, 1), desc=f"step {step_idx + 1}/{steps}")

    t = time.time()
    out = pipeline.generate(
        input_prompt=prompt,
        seed=int(seed),
        n_prompt=negative,
        sampling_steps=int(steps),
        sample_solver="default",
        width=int(width),
        height=int(height),
        guide_scale=float(guide),
        batch_size=1,
        callback=cb,
        custom_settings={"qwen21_kv_cache": "Enabled", "rgba": "Disabled"},
    )
    log(f"generated in {time.time() - t:.1f}s  shape={tuple(out.shape)} dtype={out.dtype}")
    return to_pil(out)


# ---- test pre-flight : prouve que la generation marche avant d'ouvrir l'UI ----
if not SKIP_TEST:
    try:
        log("PRE-FLIGHT: 1 image 1024x1024, 20 steps")
        img = run_one("a small dark cat walking down a narrow abandoned street",
                      steps=20, width=1024, height=1024, seed=42, guide=4.0)
        p = os.path.join(OUT_DIR, "preflight.png")
        img.save(p)
        log("PRE-FLIGHT OK ->", p, img.size, img.mode)
    except Exception:
        log("PRE-FLIGHT FAILED - on n'ouvre pas l'UI")
        traceback.print_exc()
        sys.exit(1)

if SMOKE:
    log("smoke-test termine, sortie")
    sys.exit(0)

# ---------------- Gradio UI ----------------
import gradio as gr

CSS = """
.gradio-container { max-width: 1180px !important; margin: auto !important; }
header { display: none !important; }
footer { display: none !important; }
h1 { font-weight: 700; }
"""


@torch.inference_mode()
def generate_ui(prompt, negative, steps, guide, width, height, n_images, seed, kv_cache, progress=gr.Progress()):
    gallery, status = [], "starting"
    try:
        gc.collect()
        torch.cuda.empty_cache()
        base_seed = int(seed) if seed is not None and seed >= 0 else int(torch.randint(0, 2**32 - 1, (1,)).item())
        for i in range(int(n_images)):
            yield gr.update(value=gallery, selected_index=0 if gallery else None), f"generating {i + 1}/{n_images}"
            img = run_one(prompt, steps, width, height, base_seed + i, guide,
                          negative if (negative or "").strip() else None, progress)
            gallery.append(img)
            yield gr.update(value=gallery, selected_index=0), f"{i + 1}/{n_images} done"
        zip_path = None
        if len(gallery) > 1:
            fd, zip_path = tempfile.mkstemp(suffix=".zip")
            os.close(fd)
            with zipfile.ZipFile(zip_path, "w") as z:
                for i, im in enumerate(gallery):
                    _, tmp = tempfile.mkstemp(suffix=".png")
                    os.close(_)
                    im.save(tmp)
                    z.write(tmp, f"qwen_image_{i + 1}_{base_seed + i}.png")
                    os.remove(tmp)
        yield (gr.update(value=gallery, selected_index=0),
               f"done | {len(gallery)} image(s) | seeds {list(range(base_seed, base_seed + len(gallery)))}",
               gr.update(value=zip_path, visible=zip_path is not None))
    except Exception as e:
        traceback.print_exc()
        yield gr.update(value=gallery), f"error: {e}", gr.update(visible=False)


with gr.Blocks(css=CSS, title="Qwen-Image-2.1", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Qwen-Image-2.1\n7B open-weights text-to-image. Local, private, no API key.")
    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(label="Prompt", lines=4,
                                placeholder="A red ceramic teapot on a wooden table, soft window light, detailed product photograph")
            with gr.Row():
                steps = gr.Slider(4, 50, value=20, step=1, label="Steps")
                guide = gr.Slider(0.0, 8.0, value=4.0, step=0.5, label="Guidance")
            with gr.Row():
                width = gr.Number(value=1024, label="Width (mult. of 32)", precision=0)
                height = gr.Number(value=1024, label="Height (mult. of 32)", precision=0)
            with gr.Row():
                n_images = gr.Slider(1, 4, value=1, step=1, label="Images")
                seed = gr.Number(value=-1, label="Seed (-1 random)", precision=0)
            with gr.Accordion("Advanced", open=False):
                negative = gr.Textbox(label="Negative prompt", lines=2,
                                      placeholder="low quality, blurry, distorted, bad anatomy")
                kv_cache = gr.Checkbox(value=True, label="KV cache (faster, more VRAM)")
            gen_btn = gr.Button("Generate", variant="primary", size="lg")
        with gr.Column(scale=2):
            out = gr.Gallery(label="Result", columns=2, rows=2, object_fit="contain", height=560)
            status = gr.Textbox(label="Status", interactive=False)
            zip_out = gr.File(label="Download all (zip)", visible=False)

    gen_btn.click(generate_ui,
                  inputs=[prompt, negative, steps, guide, width, height, n_images, seed, kv_cache],
                  outputs=[out, status, zip_out])

log("=" * 60)
demo.queue()
_url = demo.launch(share=True, inline=False, show_error=True)
try:
    with open(os.path.join(OUT_DIR, "gradio_url.txt"), "w") as fh:
        fh.write(str(_url))
except Exception:
    pass
log("GRADIO URL:", _url)
log("=" * 60)


## Step 5 - Lancer

Genere une image de controle, puis ouvre l'interface. Le lien Gradio s'affiche dans la sortie de cette cellule.

In [ ]:
!python -u run_qwen_image.py